# Topic 2 — Iterative Methods

Constructing the matrix representation of the Hamiltonian and diagonalizing it gives us the full information of the spectrum, which can be used to construct the dynamics.
As we have already seen, the bottleneck of this method is the memory (RAM) of a computer. To store the matrix of $n$ qubits, one needs $2^n \times 2^n = 2^{2n}$ addresses just to store the matrix. Diagonalizing this matrix will need an additional workspace of memory that is roughly that much.

On the other hand, for some problems, such as the ground state problem, we only want the eigenstate with the extreme eigenvalues. If we can somehow bypass needing to store the full $2^n \times 2^n$ matrix, and instead only store a vector of size $2^n$, we might be able to use all the available memory to store the eigenvector. Since the memory requirement drops from $\sim 4^n$ to $\sim 2^n$, the maximum accessible qubit number doubles: a machine limited to $n$ qubits by dense diagonalization can reach $2n$ qubits with a matrix-free approach.

## 1. Power method

As a warm up, we start by introducing the [power method](https://en.wikipedia.org/wiki/Power_iteration).

The Hamiltonian $H$ has eigenpairs $E_j, |E_j\rangle$ such that $H|E_j\rangle = E_j |E_j\rangle$ for $j = 1, \ldots, 2^n$. For simplicity, let us assume there is no degeneracy, and the eigenenergies are ordered by decreasing magnitude $|E_1| > |E_2| > \cdots > |E_{2^n}|$. This also means that the Hamiltonian has the eigendecomposition $H = \sum_{j} E_j |E_j\rangle \langle E_j|$.

The $k$-th power of $H$ therefore becomes
$$
H^k = \sum_{j} E_j^k |E_j\rangle \langle E_j| = E_1^k \sum_{j} r_j^k |E_j\rangle \langle E_j|,
$$
where $r_j := E_j/E_1$. From our assumption $|r_j| < 1$ for $j > 1$, so $r_j^k \rightarrow 0$ as $k \rightarrow \infty$. In other words, $H^k / E_1^k \rightarrow |E_1\rangle \langle E_1|$ becomes proportional to the projector onto the eigenstate $|E_1\rangle$ as $k \rightarrow \infty$.

This suggests the following algorithm.

### 1.1 Algorithm

**Input:** matrix-vector product $v \mapsto Hv$, initial guess $|v_0\rangle$, tolerance $\epsilon$

1. **For** $k = 0, 1, 2, \ldots$:
2. - Normalize: $|v_k\rangle \leftarrow |v_k\rangle\, /\, \| |v_k\rangle \|$
   - Apply: $|w\rangle = H|v_k\rangle$
   - Estimate eigenvalue (Rayleigh quotient): $\lambda_k = \langle v_k | w \rangle$
   - **If** $|\lambda_k - \lambda_{k-1}| < \epsilon$: **stop**; **Return** $\lambda_k,\ |v_k\rangle$
   - Set $|v_{k}\rangle \leftarrow |w\rangle $ 
3. **Return** $\lambda_k,\ |v_k\rangle$

**Convergence rate:** $\sim \left|\dfrac{E_2}{E_1}\right|^k$ — the smaller the spectral gap $|E_1| - |E_2|$, the slower the convergence.

Q: This method gives you the eigenvector with the largest magnitude eigenenergy. How to get the ground state energy using this method?

### 1.2 Memory usage

As you can see, the iterative method has the memory advantage that one can implement the algorithm without explicitly storing the whole matrix of $H$! (Or if it is sparse, we only need to store its nonzero entries.) All one needs is a function which takes a vector $|v\rangle$ and outputs $|w\rangle = H|v\rangle$.

We already have one of the ingredients in notebook `01` for this: the `apply_paulistring(paulis, qubit_labels, bitstring)` function, which lets you apply a Pauli string to a basis state $|a\rangle$, outputting the new basis state $|a'\rangle$ and its corresponding phase $\alpha$:
$$
   P|a\rangle = \alpha |a'\rangle~.
$$

Since $P$ is a linear operator, its action on any vector $|v\rangle = \sum_{a}v_a|a\rangle$ can be calculated as
$$
     P|v\rangle = \sum_{a} v_a \bigl(P|a\rangle\bigr) = \sum_{a} v_a\, \alpha(P,a)\, |a'(P,a)\rangle~.
$$

A Hamiltonian is a sum of Pauli operators with coefficients
$$
 H = \sum_{r}h_r P_r~,
$$
so its action on any vector can be calculated as
$$
     H|v\rangle = \sum_{r} h_r \bigl(P_r|v\rangle\bigr)~.
$$

The exercise below will build these operations.

The scipy ([sparse array class](https://docs.scipy.org/doc/scipy/reference/sparse.html#module-scipy.sparse)) `scipy.sparse`, `csr_matrix` and the [LinearOperator](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.LinearOperator.html) class could be useful.


### Exercise 1:

1. (This is the same as Exercise 4 in notebook 01.) Construct a function `apply_paulistring(paulis, qubit_labels, bitstring)` that takes a Pauli string, represented as a tuple of an array of `'X'`, `'Y'`, `'Z'` characters and an array of its corresponding qubit labels, and a bitstring $s$, and outputs the new bitstring $s'$ and its phase $\alpha$.

2. Use the above function to construct a function `apply_operator(operatorlist, vector)`, where `operatorlist` is a list of tuples `(coeff, paulis, qubit_labels)`.

[2'. Alternatively, construct a function `make_sparsemat(operatorlist, n_qubits)` that generates a sparse matrix from the `operatorlist`. `scipy` has a sparse matrix data type (`scipy.sparse`).]

3. Construct the transverse quantum Ising Hamiltonian with PBC for $n = 20$ qubits, with parameters $J = 1.0$ and $h = 1.5$:
$$
H = -J \sum_{j=0}^{n-1} X_j X_{j+1} - h \sum_{j=0}^{n-1} Z_j
$$
as the `operatorlist`.

[3'. Alternatively, construct a sparse matrix corresponding to the above Hamiltonian.]

4. Check `apply_operator(operatorlist, vector)` behaves correctly with the "ground truth" you established previously using small system size.

[4'. Check the sparse matrix-vector multiplication behaves correctly.]

5. Implement the power method, finding the ground state of the Hamiltonian. (Recall that it finds the largest absolute value of the eigenvalue --- how do you modify the Hamiltonian so it finds the lowest energy state?)

6. If you use the matrix-free approach, you might notice that it is not as fast as the sparse matrix approach. This would be a good topic to ask AI why and how to optimize and increase the calculation speed.

7. If one uses AI, how to check everything is correct?

In [1]:
import numpy as np

# --- Part 1: apply_paulistring (same as Exercise 4, notebook 01) ---

def apply_paulistring(paulis, qubit_labels, bitstring):
    pass


# --- Part 2: apply_operator (vectorized over all basis states) ---

def apply_operator(operatorlist, vector):
    pass

In [2]:
# --- Part 2': make_sparsemat ---
from scipy.sparse import csr_matrix

def make_sparsemat(operatorlist, n_qubits):
    pass

In [3]:
# --- Parts 3 & 4: build operatorlist, verify apply_operator against dense ---


In [4]:
# --- Part 5: power method for n=20 ---

  

## 2. Lanczos algorithm

The power method, while conceptually simple, has a convergence rate of $|E_2/E_1|^k$, which can be slow when the spectral gap $|E_1 - E_2|$ is small. Inspired by the power method, the Lanczos algorithm addresses this by building a low-dimensional subspace that captures the extreme eigenvalue structure much more efficiently.

### 2.1 Krylov subspace

Starting from an initial vector $|q_1\rangle$, repeated application of $H$ generates the **Krylov subspace** of dimension $m$:
$$
\mathcal{K}_m = \mathrm{span}\bigl\{|q_1\rangle,\ H|q_1\rangle,\ H^2|q_1\rangle,\ \ldots,\ H^{m-1}|q_1\rangle\bigr\}.
$$
The power method implicitly uses only the last vector $H^{m-1}|q_1\rangle$, discarding all previous information. The Lanczos algorithm instead builds an *orthonormal basis* $\{|q_1\rangle, \ldots, |q_m\rangle\}$ for $\mathcal{K}_m$ via the **Gram-Schmidt process**: at step $j$, apply $H$ to $|q_j\rangle$ and subtract its projections onto all previous basis vectors:
$$
|w_{j+1}\rangle = H|q_j\rangle - \sum_{k=1}^{j} \langle q_k | H | q_j \rangle\, |q_k\rangle.
$$

### 2.2 Three-term recurrence and the tridiagonal form

The sum above naively involves all $j$ previous vectors, but it collapses to just **two terms** due to the Hermiticity of $H$. For $k \leq j-2$:
$$
\langle q_k | H | q_j \rangle = \langle Hq_k | q_j \rangle,
$$
and since $H|q_k\rangle \in \mathcal{K}_{k+1} \subseteq \mathcal{K}_{j-1}$, while $|q_j\rangle \perp \mathcal{K}_{j-1}$ by construction, this inner product vanishes. The Gram-Schmidt sum therefore reduces to a **three-term recurrence**:
$$
\boxed{|w_{j+1}\rangle = H|q_j\rangle - \alpha_j\,|q_j\rangle - \beta_j\,|q_{j-1}\rangle,}
$$
where we define the **diagonal** and **off-diagonal** Lanczos coefficients:
$$
\alpha_j = \langle q_j | H | q_j \rangle, \qquad \beta_j = \langle q_{j-1} | H | q_j \rangle = \|w_j\|.
$$
(Note: $\beta_j$ is real and positive since it is the norm of $|w_j\rangle$ before normalization.)

Rearranging the recurrence gives the action of $H$ on the basis vector $|q_j\rangle$:
$$
H|q_j\rangle = \beta_j\,|q_{j-1}\rangle + \alpha_j\,|q_j\rangle + \beta_{j+1}\,|q_{j+1}\rangle.
$$
The matrix elements $T_{kj} = \langle q_k | H | q_j \rangle$ are therefore:
$$
T_{jj} = \alpha_j, \qquad T_{j\pm 1,\, j} = \beta_{j\pm 1}, \qquad T_{kj} = 0 \text{ for } |k-j|>1,
$$
so $T_m = Q_m^\dagger H Q_m$ is **tridiagonal**:
$$
T_m =
\begin{pmatrix}
\alpha_1 & \beta_2 & & \\
\beta_2 & \alpha_2 & \beta_3 & \\
& \beta_3 & \ddots & \beta_m \\
& & \beta_m & \alpha_m
\end{pmatrix}.
$$

### 2.3 Ritz values and Ritz vectors

The eigenvalues of $T_m$ are called **Ritz values** and the corresponding approximations to the eigenstates are called **Ritz vectors**. They are the best possible approximation to the eigenstates of $H$ within $\mathcal{K}_m$, as we now show.

Any vector in $\mathcal{K}_m$ can be written as $|\psi\rangle = Q_m|y\rangle$ for some coefficient vector $|y\rangle \in \mathbb{C}^m$. The energy expectation value (Rayleigh quotient) is:
$$
\frac{\langle\psi|H|\psi\rangle}{\langle\psi|\psi\rangle}
= \frac{\langle y|Q_m^\dagger H Q_m|y\rangle}{\langle y|Q_m^\dagger Q_m|y\rangle}
= \frac{\langle y|T_m|y\rangle}{\langle y|y\rangle},
$$
where we used $Q_m^\dagger Q_m = I_m$ (orthonormality of the Lanczos basis). Minimizing this over all $|y\rangle\in\mathbb{C}^m$ is exactly the eigenvalue problem for $T_m$. The minimum is the **lowest Ritz value** $\tilde{E}_0$ (the smallest eigenvalue of $T_m$), achieved by its eigenvector $|y_0\rangle$. The corresponding **Ritz vector** is:
$$
|\tilde{E}_0\rangle = Q_m|y_0\rangle \in \mathcal{K}_m.
$$
By the variational principle, $\tilde{E}_0 \geq E_0$: the Ritz value is an upper bound on the true ground state energy, and $|\tilde{E}_0\rangle$ is the state in $\mathcal{K}_m$ that minimizes the energy. As $m$ grows, $\tilde{E}_0$ decreases monotonically toward $E_0$.

### 2.4 Restart

For a finite subspace of size $m$ the ground state is generally **not** exactly contained in $\mathcal{K}_m$. However, the Ritz vector $|\tilde{E}_0\rangle$ is the best available approximation within $\mathcal{K}_m$, so it is the natural choice for the new initial state to generate a fresh Krylov subspace. Repeating this process iteratively refines the approximation.

Restarting also solves two practical problems: (1) storing all $m$ Lanczos vectors becomes expensive as $m$ grows, and (2) orthogonality among the basis vectors is gradually lost due to floating-point rounding errors. By fixing a maximum subspace size $m_{\max}$ and restarting periodically, both issues are kept under control. This is exactly the strategy implemented in `scipy.sparse.linalg.eigsh` (via ARPACK's implicitly restarted Lanczos method).

### 2.5 Algorithm

**Input:** matrix-vector product $v \mapsto Hv$, initial guess $|q_1\rangle$, number of steps $m$, tolerance $\epsilon$

1. Normalize: $|q_1\rangle \leftarrow |q_1\rangle\,/\,\||q_1\rangle\|$; set $\beta_1 = 0$, $|q_0\rangle = 0$
2. **For** $j = 1, 2, \ldots, m$:
   - Apply: $|w\rangle = H|q_j\rangle$
   - Diagonal coefficient: $\alpha_j = \langle q_j | w \rangle$
   - Orthogonalize: $|w\rangle \leftarrow |w\rangle - \alpha_j|q_j\rangle - \beta_j|q_{j-1}\rangle$
   - Off-diagonal coefficient: $\beta_{j+1} = \||w\rangle\|$
   - Normalize: $|q_{j+1}\rangle = |w\rangle\,/\,\beta_{j+1}$
   - Diagonalize $T_j$ (tridiagonal, cheap!); **if** lowest Ritz value converged to $\epsilon$: **stop**
3. **Return** lowest Ritz value $\tilde{E}_0$ and corresponding Ritz vector $Q_j|y_0\rangle$

**Convergence rate:** $\displaystyle\sim\!\left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^k$, where $\kappa = |E_{\max}/E_{\min}|$.

By replacing $\kappa$ with $\sqrt{\kappa}$ compared to the power method, Lanczos converges exponentially faster. In practice, the ground state is found in $m \ll 2^n$ iterations — `scipy.sparse.linalg.eigsh` implements exactly this algorithm (via ARPACK's implicitly restarted Lanczos).

### Exercise 2:

1. Use the Lanczos method to find the ground state of the transverse quantum Ising Hamiltonian with PBC for $n = 20$ qubits, with parameters $J = 1.0$ and $h = 1.5$:
$$
H = -J \sum_{j=0}^{n-1} X_j X_{j+1} - h \sum_{j=0}^{n-1} Z_j~.
$$
`scipy.sparse.linalg.eigsh` and the functions we built in Exercise 1 could be useful.

2. If one uses AI for 1., how to check if everything is correct?

3. In Lanczos algorithm (and power method as well), how to obtain the first excited state?

## 3. Lanczos algorithm for quantum dynamics

The Lanczos algorithm also provides a method to approximate the time-evolved state
$$
|\psi(t)\rangle = e^{-iHt}|\psi_0\rangle
$$
using iterative methods. Notice that, if we Taylor expand $e^{-iHt}$,
$$
e^{-iHt} = \sum_{k=0}^{\infty} \frac{(-it)^k}{k!} H^k,
$$
then the truncated sum $\displaystyle\sum_{k=0}^{m-1} \frac{(-it)^k}{k!} H^k |\psi_0\rangle$ lives in the Krylov subspace
$$
\mathcal{K}_m = \mathrm{span}\bigl\{|\psi_0\rangle,\ H|\psi_0\rangle,\ H^2|\psi_0\rangle,\ \ldots,\ H^{m-1}|\psi_0\rangle\bigr\}.
$$
The Krylov approach goes one step further: rather than just truncating the Taylor series, it finds the **best approximation** to $e^{-iHt}|\psi_0\rangle$ within $\mathcal{K}_m$ by working with the tridiagonal representation $T_m$ of $H$.

### 3.1 Algorithm

Assume $|\psi_0\rangle$ is normalized. Initialize the Lanczos basis with $|q_1\rangle = |\psi_0\rangle$ and run the same three-term recurrence as in Section 2.2 for $m$ steps, producing the orthonormal basis $Q_m = [|q_1\rangle\ \cdots\ |q_m\rangle]$ and the tridiagonal matrix $T_m$.

Since $|q_1\rangle = |\psi_0\rangle$, we have $|\psi_0\rangle = Q_m|e_1\rangle$ (where $|e_1\rangle$ is the first standard basis vector), so the time-evolved state is approximated as:
$$
|\psi(t)\rangle \approx Q_m\, e^{-iT_m t}\, |e_1\rangle.
$$
Diagonalizing the small $m\times m$ matrix $T_m = V\Lambda V^\dagger$ gives:
$$
e^{-iT_m t} = V\, e^{-i\Lambda t}\, V^\dagger, \qquad \bigl(e^{-i\Lambda t}\bigr)_{jj} = e^{-i\lambda_j t},
$$
so the full procedure is:

**Input:** normalized $|\psi_0\rangle$, Hamiltonian matvec $v\mapsto Hv$, time $t$, Krylov dimension $m$

1. Run $m$ Lanczos steps starting from $|q_1\rangle = |\psi_0\rangle$, accumulating $Q_m$, $\{\alpha_j\}$, $\{\beta_j\}$
2. Diagonalize $T_m$: compute $T_m = V\Lambda V^\dagger$ (cheap: $m\times m$)
3. Form the propagated coefficient vector: $|c\rangle = V\,e^{-i\Lambda t}\,V^\dagger\,|e_1\rangle$
4. **Return** $|\psi(t)\rangle \approx Q_m|c\rangle$

**Memory:** $O(m\times 2^n)$ — only the $m$ Lanczos basis vectors need to be stored.

### 3.2 Time-stepping for long-time dynamics

The Krylov approximation is most accurate for small $t$ per step. For long-time evolution, one uses **time-stepping**: divide $[0, T]$ into steps of size $\Delta t$, apply the Krylov propagator at each step using the current state as the new $|\psi_0\rangle$, and discard the Krylov basis before the next step.

$$
|\psi(0)\rangle \xrightarrow{\Delta t} |\psi(\Delta t)\rangle \xrightarrow{\Delta t} |\psi(2\Delta t)\rangle \xrightarrow{\Delta t} \cdots \xrightarrow{\Delta t} |\psi(T)\rangle
$$

At each step, a fresh $m$-dimensional Krylov subspace is built from the current state, keeping memory at $O(m\times 2^n)$ regardless of the total evolution time. A typical choice is $m \approx 10$–$30$, which already gives very accurate results for moderate $\Delta t$.

This method is known as the **Krylov subspace time evolution** (or **short iterative Lanczos**) method, and is one of the standard algorithms for quantum many-body dynamics.

### Exercise 3:

1. Implement `krylov_expm(Ham, psi0, t, m)` that approximates $e^{-iHt}|\psi_0\rangle$ using the Krylov subspace method (Section 3.1):
   - Run $m$ Lanczos steps starting from $|q_1\rangle = |\psi_0\rangle$, accumulating $Q_m$, $\{\alpha_j\}$, $\{\beta_j\}$ (detailed pseudocode below)
   - Diagonalize $T_m$ using `scipy.linalg.eigh_tridiagonal`
   - Return $Q_m V e^{-i\Lambda t} V^\dagger |e_1\rangle$

   **Lanczos steps in detail.** This is the loop of Section 2.5 with two differences: the number of steps $m$ is fixed in advance (no convergence check), and every basis vector is *stored* as a column of $Q_m$, since we need all of them at the end to map the small coefficient vector $|c\rangle$ back to the full $2^n$-dimensional space.

   **Input:** matrix-vector product $v \mapsto Hv$ (`apply_operator(Ham, v)` or sparse `Ham @ v`), initial state $|\psi_0\rangle$ of length $N = 2^n$, Krylov dimension $m$

   **Output:** $Q_m$ — an $N \times m$ complex array whose columns are $|q_1\rangle, \ldots, |q_m\rangle$; $\alpha = (\alpha_1, \ldots, \alpha_m)$ — the diagonal of $T_m$ (length $m$); $\beta = (\beta_2, \ldots, \beta_m)$ — the off-diagonal of $T_m$ (length $m-1$)

   1. Allocate: $Q_m \leftarrow$ zeros $(N \times m)$, complex; $\alpha \leftarrow$ zeros $(m)$; $\beta \leftarrow$ zeros $(m-1)$
   2. Initialize: $|q_1\rangle = |\psi_0\rangle\,/\,\||\psi_0\rangle\|$, store it as the first column of $Q_m$; set $|q_0\rangle = 0$ and $\beta_1 = 0$
   3. **For** $j = 1, 2, \ldots, m$:
      - Apply: $|w\rangle = H|q_j\rangle$ — the only step that touches $H$, and the only expensive one
      - Diagonal coefficient: $\alpha_j = \langle q_j | w \rangle$ (real up to rounding since $H$ is Hermitian; keep only the real part)
      - Orthogonalize: $|w\rangle \leftarrow |w\rangle - \alpha_j|q_j\rangle - \beta_j|q_{j-1}\rangle$ (for $j = 1$ the last term is absent)
      - *Optional but recommended:* re-orthogonalize against **all** stored vectors, $|w\rangle \leftarrow |w\rangle - \sum_{k=1}^{j} \langle q_k | w\rangle\,|q_k\rangle$, i.e. `w -= Q[:, :j] @ (Q[:, :j].conj().T @ w)`. Mathematically a no-op, numerically it cures the loss of orthogonality discussed in Section 2.4, and it costs only $O(j\, 2^n)$
      - **If** $j = m$: **break** — $|q_{m+1}\rangle$ is not part of the basis and $\beta_{m+1}$ does not appear in $T_m$
      - Off-diagonal coefficient: $\beta_{j+1} = \||w\rangle\|$
      - **If** $\beta_{j+1} < \epsilon$ (say $10^{-12}$): *happy breakdown* — $\mathcal{K}_j$ is already an invariant subspace of $H$, so the Krylov approximation is exact with $j$ vectors. Truncate: $m \leftarrow j$, keep only the first $j$ columns of $Q_m$, the first $j$ entries of $\alpha$ and the first $j-1$ entries of $\beta$, and **break**
      - Normalize: $|q_{j+1}\rangle = |w\rangle\,/\,\beta_{j+1}$, store it as column $j+1$ of $Q_m$
   4. **Return** $Q_m$, $\alpha$, $\beta$

   **Python indexing.** With 0-based arrays the natural layout is `Q[:, j]` $= |q_{j+1}\rangle$, `alpha[j]` $= \alpha_{j+1}$ and `beta[j]` $= \beta_{j+2}$, i.e. `beta[j]` couples columns `j` and `j+1`. Inside `for j in range(m)` the orthogonalization therefore subtracts `beta[j-1] * Q[:, j-1]` (skipped when `j == 0`) and the new norm goes into `beta[j]`. This is exactly the layout `scipy.linalg.eigh_tridiagonal(d, e)` expects: `d = alpha` of length $m$ and `e = beta` of length $m-1$.

2. Verify `krylov_expm` against the exact result `scipy.linalg.expm(-1j * H * t) @ psi0` for small $n$.

3. Implement `krylov_evolve(Ham, psi0, t_total, dt, m)` that performs time-stepping (Section 3.2): divide $[0, T]$ into steps of size $\Delta t$ and chain `krylov_expm` calls, returning the state at each time step.

4. Starting from the fully polarized state $|\psi_0\rangle = |\!\uparrow\uparrow\cdots\uparrow\rangle$ (the computational basis state $|11\cdots1\rangle$), compute the local magnetization
$$
\langle Z_0(t)\rangle = \langle\psi(t)|Z_0|\psi(t)\rangle
$$
as a function of time for the TFIM with $n = 20$, $J = 1.0$, $h = 1.5$, up to $T=5.0$.


In [5]:

# --- Parts 1 & 2: krylov_expm + verification ---



# --- Part 2: verify against exact matrix exponentiation for small n ---


In [6]:
# --- Parts 3 & 4: krylov_evolve + quantum quench ---
